Bronze data processing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/akhildataengineer@hotmail.com/Atlikon_migration_pipeline/1_setup/3_utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-store-dp1/{data_source}/*.csv' #s3://sportsbar-store-dp/customers/*.csv
base_path

In [0]:
dbutils.fs.ls('s3://sportsbar-store-dp1/')

In [0]:
df_raw = spark.read\
        .format('csv')\
        .option('header', True)\
        .option('inferSchema', True)\
        .load(base_path)\
        .withColumn("read_timestamp", F.current_timestamp())\
        .select("*", "_metadata.file_name", "_metadata.file_size")

display(df_raw.limit(10))

In [0]:
df_raw.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

Silver Data Processing

In [0]:
# Drop duplicates
print(f"Rows before drop: {df_raw.count()}")
df_enr = df_raw.dropDuplicates(["product_id"])
print(f"Rows after drop: {df_enr.count()}")

In [0]:
# Title case fix
df_enr.select("category").distinct().show()

df_enr = df_enr.withColumn("category", F.when(F.col("category").isNull(), None).otherwise(F.initcap(F.col("category"))))

In [0]:
# Replace protien to protein
df_enr = df_enr.withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)protien", "Protein"))\
                .withColumn("category", F.regexp_replace(F.col("category"), "(?i)protien", "Protein"))

In [0]:
# Create division column 
df_enr = df_enr\
            .withColumn("division", F.when(F.col("category") == "Energy Bar", "Nutrition Bars")\
                                    .when(F.col("category") =="Protein Bars", "Nutrition Bars")\
                                    .when(F.col("category") == "Granola & Cereals", "Breakfast Foods")\
                                    .when(F.col("category") == "Recovery Drinks", "Dairy & Recovery")\
                                    .when(F.col("category") == "Healthy Snacks", "Hea;thy Snacks")\
                                    .when(F.col("category") == "Electrolyte Mix", "Hydration & Electrolytes")
                                    .otherwise("Other"))

In [0]:
# Quantity/Weight from string
df_enr = df_enr\
            .withColumn("variant", F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1))
display(df_enr.limit(10))

In [0]:
# Product code 
df_enr = df_enr\
            .withColumn("product_code", F.sha2(F.col("product_id").cast("string"), 256))\
            .withColumn("product_id", F.when(
                                        F.col("product_id").cast("string").rlike("^[0-9]+$"),
                                        F.col("product_id").cast("string"))\
                                        .otherwise(F.lit(999999).cast("string")))\
            .withColumnRenamed("product_name", "product")

display(df_enr.limit(10))

In [0]:
# Reorder columns
df_enr = df_enr.select("product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "_metadata.file_name", "_metadata.file_size")
display(df_enr.limit(1))

In [0]:
df_enr.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("overwriteSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

Gold Data Processing

In [0]:
df_enr = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")
df_agg = df_enr.select("product_code", "product_id", "division", "category", "product", "variant")
df_agg.show(5)

In [0]:
df_agg\
  .write\
  .format("delta")\
  .option("delta.enableChangeDataFeed", "true")\
  .option("mergeSchema", "true")\
  .mode("overwrite")\
  .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

Merge with Parent table - Atlikon

In [0]:
delta_table = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.dim_{data_source}")
df_child_products = spark.sql(f"SELECT product_code, division, category, product, variant FROM {catalog}.{gold_schema}.sb_dim_{data_source};")
df_child_products.show(5)

In [0]:
# Upsert
delta_table.alias("t")\
        .merge(
            source = df_child_products.alias("s"),
            condition = "t.product_code = s.product_code"
        )\
        .whenMatchedUpdate(
            set = {
                "t.division": "s.division",
                "t.category": "s.category",
                "t.product": "s.product",
                "t.variant": "s.variant"
            }
        )\
        .whenNotMatchedInsert(
            values = {
                "t.product_code": "s.product_code",
                "t.division": "s.division",
                "t.category": "s.category",
                "t.product": "s.product",
                "t.variant": "s.variant"
            }
        )\
        .execute()